In [ ]:
'''
Hi bro, this is an auto system for extract data (name, emails of agents),
then it automatically send email to them. I also had sent this to the CTO. Instead of one, I implented 3 different
style/content for email template (it's like an A/B test), so we can identify which
style/content has more reply

Additional info:
it will extract the REINSW web, but incase the system is extract unnecessary data, I
implented an additonal function, so you mannually enter the target postcode, then the agent's data
will pop up, if can check first is these the data you want, if yes, just go back to python
and press 'enter' on the keyboard, then it will start running. (By adding this function, you
dont need to fix the code, you just need to operate on the web which is more readable and easier)

We might need to use gmail instead of outlook, cuz it's more complicated for my to connect outlook,
so I checked with the CTO, he said no need of using outlook or remove the auto email function
tell someone to mannually send it? He say up to you, maybe you double check with him?

There will be some basic instruction below too, but let me know if you have any problem with it or
if you want I run the code for you?
'''
import csv
import time
import smtplib
import ssl
from email.message import EmailMessage
from selenium import webdriver
from selenium.webdriver.common.by import By

# Mannual enter your email between " "
SENDER_EMAIL = "aaaaaaaaaaa@gmail.com"
SENDER_PASSWORD = "your 16 digit app password"
# Use an App Password, not your real password!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# I will send u a pdf how to get that

# email template
TEMPLATE_0 = """Hi {name},

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
I have manually removed some of the content here to respect my previous company's
privacy and proprietary rights.
"""

TEMPLATE_1 = """Hello {name},

yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
I have manually removed some of the content here to respect my previous company's
privacy and proprietary rights."""

TEMPLATE_2 = """Dear {name},

zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz
I have manually removed some of the content here to respect my previous company's
privacy and proprietary rights."""

TEMPLATES = [TEMPLATE_0, TEMPLATE_1, TEMPLATE_2]
SUBJECTS = ["x", "y", "z"]
# I have manually removed some of the content here to respect my previous company's
# privacy and proprietary rights

# ______________________________________________________________________

# scraper
def extract_links_from_context(driver_context):
    urls = []
    anchors = driver_context.find_elements(By.XPATH, "//*[contains(text(), 'AUSTRALIA')]")
    for anchor in anchors:
        try:
            card = anchor.find_element(By.XPATH, "./ancestor::*[.//a][1]")
            links = card.find_elements(By.TAG_NAME, "a")
            for a in links:
                href = a.get_attribute("href")
                if href and href.startswith("http") and "maps.google" not in href and href not in urls:
                    urls.append(href)
        except:
            continue
    return urls

def send_grouped_emails(agent_data):
    # data cleaning - filter out agents who didnt have an email listed
    valid_agents = [agent for agent in agent_data if agent[1] != "N/A"]

    if not valid_agents:
        print("\n No valid emails")
        return

    print(f"\n" + "="*50)
    print(f"EMAIL SYSTEM: Found {len(valid_agents)} valid agent emails.")
    print("Group 1 will receive Template 1")
    print("Group 2 will receive Template 2")
    print("Group 3 will receive Template 3")


    confirm = input("Would you like to connect to the email server and send these now? (type 'yes' or 'no'): ")
    if confirm.lower() != 'yes':
        print("Aborting email send!!!!!!")
        return

    print("Connecting to email server ><")
    context = ssl.create_default_context()

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            print("Successfully connected to email server!\n")

            for index, agent in enumerate(valid_agents):
                name, email = agent[0], agent[1]

                # we split the grps
                group_index = index % 3

                msg = EmailMessage()
                msg.set_content(TEMPLATES[group_index].format(name=name))
                msg['Subject'] = SUBJECTS[group_index]
                msg['From'] = SENDER_EMAIL
                msg['To'] = email

                print(f"Sending Template {group_index + 1} to {name} ({email})...")
                server.send_message(msg)

                # (avoid spam filters)
                time.sleep(2)

        print("\n All emails sent")

    except Exception as e:
        print(f"\n Email Error: {e}")
        print("Did you remember to use an App Password if using Gmail + let me know if this keep showing up")

def run_scraper():
    driver = webdriver.Chrome()
    url = "https://www.reinsw.com.au/Web/Web/Find_an_Agent/find_a_reinsw_member-agent.aspx"
    driver.get(url)

    print("\n" + "="*50)
    print("a) Once the grid of agents appears...")
    input("b) RETURN here and press ENTER --> if you dont press ENTER it will nottttttt runnnnnnnnnn.")
    print("="*50 + "\n")

    print("Scanning for agents (would take some time)")
    agent_urls = extract_links_from_context(driver)
    found_in_iframe = -1

    if not agent_urls:
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        for i in range(len(iframes)):
            driver.switch_to.frame(i)
            agent_urls = extract_links_from_context(driver)
            if agent_urls:
                found_in_iframe = i
                break
            driver.switch_to.default_content()

    if not agent_urls:
        print("Found 0 agents (let me know if this happened)")
        driver.quit()
        return

    print(f"Found {len(agent_urls)} unique profile links! Starting extraction...")
    all_data = []
    main_window = driver.current_window_handle

    for link in agent_urls:
        try:
            driver.execute_script(f"window.open('{link}', '_blank');")
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(2)

            try:
                name = driver.find_element(By.TAG_NAME, "h1").text
            except:
                try:
                    name = driver.find_element(By.XPATH, "//h2 | //div[contains(@class, 'Name')]").text
                except:
                    name = "Unknown"

            try:
                email_elem = driver.find_element(By.XPATH, "//a[starts-with(@href, 'mailto:')]")
                email = email_elem.get_attribute("href").replace("mailto:", "")
            except:
                email = "N/A"

            print(f"Scraped: {name} -> {email}")
            all_data.append([name, email])

            driver.close()
            driver.switch_to.window(main_window)
            if found_in_iframe != -1:
                driver.switch_to.frame(found_in_iframe)

        except Exception as e:
            print(f"Error on profile {link}: {e}")
            if len(driver.window_handles) > 1:
                driver.close()
                driver.switch_to.window(main_window)
                if found_in_iframe != -1:
                    driver.switch_to.frame(found_in_iframe)

    # save (CSV)
    if all_data:
        with open('promptlaw_agents_data.csv', 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["Name", "Email"])
            writer.writerows(all_data)
        print(f"\n SUCCESS! Saved {len(all_data)} promptlaw_agents_data.csv")

        send_grouped_emails(all_data)

    driver.quit()

if __name__ == "__main__":
    run_scraper()